# Ingest sprints.json file
1. Read the all the files from the sprints folder using spark dataframe reader API
1. Define and enforce schema 
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

> Note: JSON is in multi line format

In [0]:
%run  ../common/01_Environmnet_config

In [0]:
%run  ../common/02_bronze_helpers

In [0]:
source_File = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"
print(source_File)
print(table_name)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, FloatType

sprints_schema = StructType([
    StructField('date', DateType(), True),
    StructField('raceName', StringType(), True),
    StructField('round', IntegerType(), True),
    StructField('season', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('constructorId', StringType(), True),
    StructField('driverId', StringType(), True),
    StructField('grid', IntegerType(), True),
    StructField('laps', IntegerType(), True),
    StructField('number', IntegerType(), True),
    StructField('points', FloatType(), True),
    StructField('position', IntegerType(), True),
    StructField('positionText', StringType(), True),
    StructField('status', StringType(), True),
])

In [0]:
sprints_df = (
    spark.read.format("json")
    .schema(sprints_schema)
    .option("mode", "FAILFAST")
    .option('multiLine', "true")
    .load(source_File)
)

In [0]:
display(sprints_df)

In [0]:
sprints_final_df = add_ingestion_medatat(sprints_df)

In [0]:
(
    sprints_final_df
                .write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name)
    
)

In [0]:
spark.sql(F"select * from {table_name}").display()

In [0]:
%sql
select season, Count(*) 
from formula1.bronze.sprints
group by season
order by season